# Diplomski rad - Primena transformer arhitekture neuronskih mreza za detekciju DoS napada

## Beleznica 2: Priprema podataka (Preprocessing)

Cilj ove beleznice je ciscenje i priprema podataka za treniranje modela: uklanjanje neregularnih instanci, enkodiranje labela, podela na train/test skup, uklanjanje atributa bez varijanse i skaliranje atributa. Podaci se ucitavaju iz parquet fajla kreiranog u prethodnoj beleznici.

### 1. Ucitavanje podataka

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_path = DATA_DIR / "raw_dos_wednesday.parquet"
df = pd.read_parquet(data_path)

label_col = "Label"

print("Shape:", df.shape)
print(df[label_col].value_counts())


Mounted at /content/drive
Shape: (584991, 78)
Label
Benign              391235
DoS Hulk            172846
DoS GoldenEye        10286
DoS slowloris         5385
DoS Slowhttptest      5228
Heartbleed              11
Name: count, dtype: int64


### 2. Uklanjanje Heartbleed instanci

Klasa Heartbleed ne pripada DoS kategoriji napada, s obzirom na to da se zasniva na drugacijem mehanizmu (eksploatacija ranjivosti u OpenSSL biblioteci, a ne preopterecenje resursa), a dodatno sadrzi svega 11 instanci, sto je nedovoljno za pouzdano treniranje modela. Ove instance se uklanjaju iz dataseta.

In [ ]:
print("Pre uklanjanja:", df.shape)

df = df[df[label_col] != 'Heartbleed'].copy()

print("Posle uklanjanja:", df.shape)
print()
print(df[label_col].value_counts())


Pre uklanjanja: (584991, 78)
Posle uklanjanja: (584980, 78)

Label
Benign              391235
DoS Hulk            172846
DoS GoldenEye        10286
DoS slowloris         5385
DoS Slowhttptest      5228
Heartbleed               0
Name: count, dtype: int64


### 3. Provera NaN i beskonacnih vrednosti

Pojedini atributi (na primer, brzina protoka podataka) racunaju se deljenjem, sto moze proizvesti beskonacnu vrednost u slucaju da je trajanje mreznog toka jednako nuli. Neuronske mreze ne mogu da obradjuju nedostajuce (NaN) ili beskonacne (Infinity) vrednosti, pa se ovakve instance identifikuju i uklanjaju iz dataseta.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

n_nan = df[numeric_cols].isna().sum().sum()
n_inf = np.isinf(df[numeric_cols]).sum().sum()

print(f"Broj NaN vrednosti: {n_nan}")
print(f"Broj Infinity vrednosti: {n_inf}")

df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

print("\nPre uklanjanja:", df.shape)
df = df.dropna().copy()
print("Posle uklanjanja:", df.shape)


Broj NaN vrednosti: 0
Broj Infinity vrednosti: 0

Pre uklanjanja: (584980, 78)
Posle uklanjanja: (584980, 78)


### 4. Provera duplikata

Proverava se prisustvo potpuno identicnih redova (duplikata), koji mogu vestacki uvecati zastupljenost odredjenih obrazaca u skupu podataka. Nakon provere, eventualni duplikati se uklanjaju.

In [ ]:
n_dup = df.duplicated().sum()
print(f"Broj dupliranih redova: {n_dup}")

df = df.drop_duplicates().copy()
print(f"Shape posle uklanjanja duplikata: {df.shape}")


Broj dupliranih redova: 0
Shape posle uklanjanja duplikata: (584980, 78)

Kolone bez varijanse (10): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Shape posle uklanjanja: (584980, 68)


### 5. Enkodiranje labela

S obzirom na to da model za klasifikaciju zahteva numeričku reprezentaciju ciljnih klasa, kategorička kolona Label transformiše se korišćenjem klase LabelEncoder iz biblioteke scikit-learn. Dobijeno mapiranje klasa cuva se radi tumacenja rezultata u kasnijim fazama istrazivanja.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df[label_col])

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Mapiranje klasa:")
for name, code_ in label_mapping.items():
    print(f"  {code_} -> {name}")

print("\nRaspodela nakon enkodiranja:")
print(df['Label_encoded'].value_counts().sort_index())


Mapiranje klasa:
  0 -> Benign
  1 -> DoS GoldenEye
  2 -> DoS Hulk
  3 -> DoS Slowhttptest
  4 -> DoS slowloris

Raspodela nakon enkodiranja:
Label_encoded
0    391235
1     10286
2    172846
3      5228
4      5385
Name: count, dtype: int64


### 6. Podela podataka i uklanjanje atributa bez varijanse

Skup podataka se deli na dva dela: trening skup, na kojem se model obučava, i test skup, koji služi da se proveri koliko uspešno model primenjuje stečeno znanje na podatke koje nije video tokom treniranja. Koristi se stratifikovana podela, koja obezbeđuje da odnos klasa ostane približno jednak u oba skupa, što je od posebnog značaja s obzirom na uočenu neizbalansiranost klasa.

Nakon podele, na osnovu trening skupa identifikuju se atributi bez varijanse, odnosno atributi koji imaju istu vrednost u svim trening instancama. Identifikovani atributi zatim se uklanjaju iz trening i test skupa, čime se sprečava korišćenje informacija iz test skupa pri donošenju odluka o predobradi podataka.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c not in [label_col, 'Label_encoded']]

X = df[feature_cols]
y = df['Label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

zero_var_cols = [
    c for c in X_train.columns
    if X_train[c].nunique() == 1
]

print(f"Kolone bez varijanse u trening skupu ({len(zero_var_cols)}):")
print(zero_var_cols)

X_train = X_train.drop(columns=zero_var_cols)
X_test = X_test.drop(columns=zero_var_cols)

feature_cols = X_train.columns.tolist()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print()
print("Raspodela klasa u train skupu:")
print(y_train.value_counts().sort_index())
print()
print("Raspodela klasa u test skupu:")
print(y_test.value_counts().sort_index())


Train shape: (467984, 67)
Test shape: (116996, 67)

Raspodela klasa u train skupu:
Label_encoded
0    312988
1      8229
2    138277
3      4182
4      4308
Name: count, dtype: int64

Raspodela klasa u test skupu:
Label_encoded
0    78247
1     2057
2    34569
3     1046
4     1077
Name: count, dtype: int64


### 6.1 Pregled preostalih atributa

Nakon uklanjanja atributa bez varijanse, prikazuje se konacna lista od 67 numerickih atributa koji se koriste za treniranje i evaluaciju modela.

In [ ]:
print(f"Broj feature-a: {len(feature_cols)}")
print("Preostali atributi:")

for i, col in enumerate(feature_cols, 1):
    print(f"{i}. {col}")

### 7. Skaliranje atributa

Numericki atributi skaliraju se koriscenjem StandardScaler transformacije, kojom se svaki atribut transformise tako da ima srednju vrednost 0 i standardnu devijaciju 1. Skaliranje je neophodno s obzirom na to da atributi u izvornom obliku poseduju bitno razlicite opsege vrednosti, sto bi bez odgovarajuce normalizacije moglo dovesti do toga da neuronska mreza pridaje neopravdano veci znacaj atributima sa vecim brojcanim vrednostima.

Parametri transformacije (srednja vrednost i standardna devijacija) odredjuju se iskljucivo na osnovu skupa za treniranje, nakon cega se ista transformacija primenjuje i na skup za testiranje - cime se sprecava curenje informacija iz skupa za testiranje u proces pripreme podataka.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Primer pre skaliranja (prvi red, prvih 5 feature-a):")
print(X_train.iloc[0, :5].values)

print("\nPrimer posle skaliranja (prvi red, prvih 5 feature-a):")
print(X_train_scaled[0, :5])


Primer pre skaliranja (prvi red, prvih 5 feature-a):
[6.000000e+00 7.248398e+06 6.000000e+00 6.000000e+00 2.332000e+03]

Primer posle skaliranja (prvi red, prvih 5 feature-a):
[-0.69355027 -0.57974036 -0.00622395 -0.0058213   0.24995469]


### 8. Cuvanje pripremljenih podataka

Pripremljeni skupovi podataka (skalirani, sa numerickim labelama), zajedno sa parametrima transformacije, mapiranjem klasa i spiskom atributa, cuvaju se u `data` direktorijumu projekta radi upotrebe u narednim beleznicama, bez potrebe za ponavljanjem celog procesa pripreme podataka.

In [ ]:
import joblib

np.save(DATA_DIR / "X_train.npy", X_train_scaled)
np.save(DATA_DIR / "X_test.npy", X_test_scaled)
np.save(DATA_DIR / "y_train.npy", y_train.values)
np.save(DATA_DIR / "y_test.npy", y_test.values)

joblib.dump(scaler, DATA_DIR / "scaler.pkl")
joblib.dump(label_mapping, DATA_DIR / "label_mapping.pkl")
joblib.dump(feature_cols, DATA_DIR / "feature_cols.pkl")

print("Sacuvano:")
print(" - X_train.npy, X_test.npy, y_train.npy, y_test.npy")
print(" - scaler.pkl")
print(" - label_mapping.pkl")
print(" - feature_cols.pkl")


Sacuvano:
 - X_train.npy, X_test.npy, y_train.npy, y_test.npy
 - scaler.pkl (za buduce nove podatke)
 - label_mapping.pkl (mapiranje broj->naziv klase)
 - feature_cols.pkl (imena feature-a)


## Zakljucak beleznice

Nakon uklanjanja instanci klase Heartbleed, provere nedostajucih i beskonacnih vrednosti (nijedna nije pronadjena), uklanjanja duplikata (nijedan nije pronadjen) i deset atributa bez varijanse, konacan skup podataka sadrzi 584.980 instanci opisanih sa 67 numerickih atributa i pet relevantnih klasa. Podaci su podeljeni na skup za treniranje (467.984 instance) i skup za testiranje (116.996 instanci) uz ocuvanje proporcije klasa, i skalirani StandardScaler transformacijom, prilagodjenom iskljucivo na skupu za treniranje.

**Sledeci korak:** Beleznica `03_random_forest_baseline.ipynb`, u okviru koje se implementira i evaluira Random Forest model kao bazni model za poredjenje sa transformer arhitekturom.